# 03 - Silver Layer: Customer Dimension (SCD Type 2)

In [0]:


from pyspark.sql.functions import *
from delta.tables import DeltaTable

dbutils.widgets.text("catalog","delta_catalog")
dbutils.widgets.text("schema",  "delta_demo")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")




## Create SCD2 table (only if not exists — preserves history)



In [0]:



spark.sql(f"""
          create  table if not exists {CATALOG}.{SCHEMA}.silver_customers
          (
            customer_id int,
            name string,
            city string,
            tier string,
            email string,
            is_current boolean,
            start_date date,
            end_date date
          )
          using delta
          """)

## Initial load (only if table is empty)


In [0]:
current_count=spark.table(f"{CATALOG}.{SCHEMA}.silver_customers").count()

if current_count==0:
    print("Empty silver_customers-doing initial load")
    customers_raw = spark.table(f"{CATALOG}.{SCHEMA}.landing_customers")

    customers_scd=customers_raw\
        .withColumn("customer_id",col("customer_id").cast("int"))\
         .withColumn("is_current",lit(True))\
             .withColumn("start_date",current_date())\
                 .withColumn("end_date",lit(None).cast("date"))
    customers_scd.write.format("delta")\
    .mode("append")\
        .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_customers")
    print("Initial load done:", spark.table(f"{CATALOG}.{SCHEMA}.silver_customers").count())
else:
    print(f"Silver customers exists with {current_count} rows — skipping init")

## Apply SCD2 updates

In [0]:
updates = spark.createDataFrame([
    (0,  "Customer_0",  "Kolkata", "Platinum", "customer_0@email.com"),
    (5,  "Customer_5",  "Delhi",   "Gold",     "customer_5@email.com"),
    (10, "Customer_10", "Mumbai",  "Bronze",   "customer_10@email.com"),
], ["customer_id", "name", "city", "tier", "email"])

updates = updates.withColumn("customer_id", col("customer_id").cast("int"))   # ← ADD THIS

target = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.silver_customers")



In [0]:
updates = spark.createDataFrame([
    (0,  "Customer_0",  "Kolkata", "Platinum", "customer_0@email.com"),
    (5,  "Customer_5",  "Delhi",   "Gold",     "customer_5@email.com"),
    (10, "Customer_10", "Mumbai",  "Bronze",   "customer_10@email.com"),
], ["customer_id", "name", "city", "tier", "email"])

updates = updates.withColumn("customer_id", col("customer_id").cast("int"))

target = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.silver_customers")

# Step 1: Expire changed records
target.alias("t").merge(
    updates.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true AND t.city != s.city"
).whenMatchedUpdate(set={
    "is_current": "false",
    "end_date":   "current_date()"
}).execute()

# Step 2: Insert new versions for changed customers
changed_ids = updates.alias("s").join(
    spark.table(f"{CATALOG}.{SCHEMA}.silver_customers").alias("t"),
    (col("s.customer_id") == col("t.customer_id")) & 
    (col("t.end_date") == current_date()),
    "inner"
).select("s.*")

changed_ids.withColumn("is_current", lit(True))\
    .withColumn("start_date", current_date())\
    .withColumn("end_date",   lit(None).cast("date"))\
    .write.format("delta")\
    .mode("append")\
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_customers")

print("SCD2 applied. Customer 0 history:")
spark.table(f"{CATALOG}.{SCHEMA}.silver_customers")\
     .filter("customer_id = 0").show()

dbutils.notebook.exit("03_silver_customers: SUCCESS")